# 職場の事務職員 離職シミュレーション（ローカル実行用）

財務・人事部門から取得した**集計表**（`workplace/roster_template.csv` 様式）から
合成コホートを構築し、JPSED で学習した較正済み離職モデルで数年先まで前進させる。

- **データは手元から出ません**（すべてローカル/自分のDriveで完結）
- 職場版の定義：離職＝職場からの退出（再就業ステップは使わない）。
  **定年退職は年齢から確定的に**扱い、中途離職と分けて集計する
- シナリオ：賃金ルール（ベアゼロ＝定昇のみ vs モデル賃金）× 将来インフレ率

必要ファイル：JPSED データ（学習用）、記入済み roster CSV。

In [ ]:
# ── 設定 ──────────────────────────────────────────────────────────────
USE_SYNTHETIC   = False   # True: 同梱合成データで動作確認（実推計は False）
JPSED_DATA_DIR  = '/path/to/JPSED/data'          # 実データの場所（local か Drive）
ROSTER_CSV      = 'workplace/roster_example.csv' # 記入済み集計表（まずは記入例で試走可）

BASE_YEAR       = 2025    # 集計表の基準年（この年の12月在籍と解釈）
YEARS           = 5       # 何年先まで回すか
N_SIMS          = 300     # モンテカルロ反復数
RETIREMENT_AGE  = 65      # 定年（None で無効化）
INDUSTRY_CODE   = 55      # 教育（小中高・短大・大学等）
OCCUPATION_CODE = 70      # その他一般事務系職（管理事務なら 56）

# 賃金シナリオ：'fixed' = 定昇のみ（ベアゼロ）, 'model' = 学習した賃金モデル
WAGE_MODE       = 'fixed'
TEISHO_RATE     = 0.017   # 定昇のみの名目昇給率（例 1.7%）
FUTURE_INFLATION_PCT = 2.0  # 将来の年インフレ率の想定（%）

import os, sys
if os.path.basename(os.getcwd()) in ('notebooks','workplace'): os.chdir('..')
sys.path.insert(0, os.getcwd())

## 1. JPSED で離職・賃金モデルを学習（全雇用者・較正済み）

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data.loader import DataLoader
from src.data.panel_builder import PanelBuilder
from src.data.transitions import TransitionFrameBuilder
from src.data.macro import MacroData
from src.features.assembler import FeatureAssembler
from src.models.transitions import TransitionModels

data_dir = 'data/synthetic' if USE_SYNTHETIC else JPSED_DATA_DIR
pb  = PanelBuilder(DataLoader(data_dir, synthetic=USE_SYNTHETIC))
fa  = FeatureAssembler()
mac = MacroData()
tf, FEATS = TransitionFrameBuilder(pb, fa).build()
tm = TransitionModels(FEATS).fit(tf)          # 全期間で学習（推計用）
print('trained on', len(tf), 'transitions | separation base rate %.3f' % tf['separated'].mean())

## 2. 集計表 → 合成コホート

バンド内は一様サンプリング。給与台帳にない項目（学歴・家族・労働時間）は
**欠測(NaN)のまま**投入する（HistGBM が欠測を分岐でそのまま扱えるため）。

In [ ]:
EMP_MAP = {'正規':1, 'パート':2, '派遣':3, '契約':4, '嘱託':5}
SEX_MAP = {'男':1, '女':2, '不明':np.nan}

def parse_band(s):
    a,b = str(s).split('-'); return float(a), float(b)

def expand_roster(path, seed=12345):
    r = pd.read_csv(path); rng = np.random.default_rng(seed); rows=[]
    for _, row in r.iterrows():
        n = int(row['headcount'])
        if n<=0: continue
        a0,a1 = parse_band(row['age_band']); t0,t1 = parse_band(row['tenure_band'])
        s0,s1 = parse_band(row['salary_band_manen'])
        for _ in range(n):
            age    = rng.uniform(a0, a1+1)
            tenure = min(rng.uniform(t0, t1+1), max(age-18,0))
            emp    = EMP_MAP[str(row['employment_type']).strip()]
            rows.append(dict(
                gender=SEX_MAP.get(str(row['sex']).strip(), np.nan),
                age=age, education=np.nan, has_spouse=np.nan, has_child=np.nan,
                num_children=np.nan, youngest_child_age=np.nan,
                contract_type=emp, industry=INDUSTRY_CODE, firm_size=np.nan,
                occupation=OCCUPATION_CODE, position=np.nan, weekly_hours=np.nan,
                is_regular=int(emp==1), tenure_years=tenure,
                annual_income=rng.uniform(s0, s1),
                prev_annual_income=np.nan, salary_growth_rate=np.nan,
            ))
    return pd.DataFrame(rows)

cohort0 = expand_roster(ROSTER_CSV)
print('cohort size:', len(cohort0))
cohort0.groupby('contract_type').agg(n=('age','size'), mean_age=('age','mean'),
                                     mean_income=('annual_income','mean')).round(1)

## 3. 将来CPI経路と初期のマクロ・実質項

In [ ]:
cpi_hist = mac.lookup()['cpi_all_items']
def build_cpi_path(base_year, years, infl_pct):
    path={}
    last_year = int(cpi_hist.index.max()); last = float(cpi_hist.loc[last_year])
    for y in range(min(base_year-1, int(cpi_hist.index.min())), base_year+years+1):
        if y in cpi_hist.index: path[y]=float(cpi_hist.loc[y]); last=path[y]
        elif y>last_year:
            last = last*(1+infl_pct/100); path[y]=last
    return path
CPI = build_cpi_path(BASE_YEAR, YEARS, FUTURE_INFLATION_PCT)

def infl(y): return (CPI[y]/CPI[y-1]-1)*100
ref0 = BASE_YEAR-1   # 基準年の年収は暦年 BASE_YEAR-1 のものとみなす（JPSED流儀）
cohort0['cpi'] = CPI[ref0]; cohort0['inflation_rate'] = infl(ref0)
cohort0['real_annual_income'] = cohort0['annual_income']/(CPI[ref0]/100)
cohort0['real_prev_annual_income']=np.nan; cohort0['real_salary_growth_rate']=np.nan
print({y: round(v,1) for y,v in CPI.items() if y>=ref0})

## 4. 職場シミュレーション

各年：①定年退出（年齢≥定年, 確定的）→ ②中途離職の抽選 Bernoulli($p_s$) →
③残留者の更新（年齢+1・勤続+1・賃金ルール適用・実質化）。

In [ ]:
def simulate_workplace(cohort_init, years=YEARS, n_sims=N_SIMS, wage_mode=WAGE_MODE,
                       teisho=TEISHO_RATE, retirement_age=RETIREMENT_AGE, seed0=0):
    N0=len(cohort_init); recs=[]
    for s in range(n_sims):
        rng=np.random.default_rng(seed0+s)
        st=cohort_init.copy()
        for k in range(1, years+1):
            year=BASE_YEAR+k-1; ref=year   # 遷移 year->year+1（暦年 year の出来事）
            if retirement_age is not None and len(st)>0:
                retire = st['age']>=retirement_age
            else:
                retire = pd.Series(False, index=st.index)
            n_ret=int(retire.sum()); st=st[~retire]
            if len(st)>0:
                p=tm.p_separate(st[FEATS])
                quit_=rng.random(len(st))<p
                n_quit=int(quit_.sum()); st=st[~quit_]
            else: n_quit=0
            if len(st)>0:
                old_nom=st['annual_income'].to_numpy(float)
                new_nom = old_nom*(1+teisho) if wage_mode=='fixed' else tm.wage_stay(st[FEATS])
                cpi=CPI[ref]
                st=st.assign(age=st['age']+1, tenure_years=st['tenure_years']+1,
                    prev_annual_income=old_nom, annual_income=new_nom,
                    salary_growth_rate=np.clip((new_nom-old_nom)/old_nom,-1,3),
                    cpi=cpi, inflation_rate=infl(ref),
                    real_prev_annual_income=st['real_annual_income'],
                    real_annual_income=new_nom/(cpi/100))
                st=st.assign(real_salary_growth_rate=np.clip(
                    (st['real_annual_income']-st['real_prev_annual_income'])
                    /st['real_prev_annual_income'],-1,3))
            recs.append(dict(sim=s, k=k, year=year+1, remain=len(st),
                             quits=n_quit, retirements=n_ret))
    return pd.DataFrame(recs), N0

res, N0 = simulate_workplace(cohort0)
g=res.groupby('k')
summary=pd.DataFrame({
    'year': g['year'].first(),
    'headcount_index_mean': g['remain'].mean()/N0*100,
    'index_p5':  g['remain'].quantile(.05)/N0*100,
    'index_p95': g['remain'].quantile(.95)/N0*100,
    'quits_mean': g['quits'].mean(),
    'retirements_mean': g['retirements'].mean(),
}).round(2)
print(f'初期人数 = {N0}（=100 と基準化）  賃金モード={WAGE_MODE}, 将来インフレ={FUTURE_INFLATION_PCT}%')
summary

## 5. 可視化：従業員数指数（現在=100）の推移

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
ks=summary.index.to_numpy()
ax[0].plot(np.r_[0,ks], np.r_[100,summary['headcount_index_mean']], 'o-', color='#0072B2')
ax[0].fill_between(ks, summary['index_p5'], summary['index_p95'], alpha=.2, color='#0072B2')
ax[0].set_xlabel('years ahead'); ax[0].set_ylabel('headcount index (now=100)')
ax[0].set_title('Workforce index (90% band)')
ax[1].bar(ks-0.2, summary['quits_mean'], width=.4, label='quits (model)', color='#D55E00')
ax[1].bar(ks+0.2, summary['retirements_mean'], width=.4, label='retirements (age rule)', color='#999999')
ax[1].set_xlabel('years ahead'); ax[1].set_ylabel('expected leavers / year'); ax[1].legend()
ax[1].set_title('Leavers: mid-career quits vs mandatory retirement')
plt.tight_layout(); plt.show()

## 6. シナリオ比較：ベアゼロ vs モデル賃金

In [ ]:
rows=[]
for mode,label in [('fixed','ベアゼロ（定昇のみ）'), ('model','モデル賃金（過去平均並みの昇給）')]:
    r,_=simulate_workplace(cohort0, wage_mode=mode)
    m=r.groupby('k')['remain'].mean()/N0*100
    rows.append(pd.Series(m.values, index=m.index, name=label))
comp=pd.DataFrame(rows).T; comp.index.name='years ahead'
display(comp.round(2))
comp.plot(marker='o', figsize=(6,4), ylabel='headcount index (now=100)')
plt.title('Wage-rule scenarios'); plt.tight_layout(); plt.show()

## 注意（解釈のガードレール）

- 学習は JPSED 全雇用者。職場固有の要因は業種・職種・年齢・勤続・賃金等の
  **観測変数を通じてのみ**反映される（貴機関の固有文化は入らない）
- 集計表からの合成なので個人は再現していない。**組織水準の期待値と分布幅**が
  読むべき出力（小規模組織では実現値のブレが大きい）
- 定年は年齢ルールで確定処理。再雇用制度（定年後継続）を入れる場合は
  retirement で除く代わりに嘱託へ転換させる改修が可能
- 補充採用は入れていない（純減の推移）。採用計画を重ねる場合は毎年
  新規行を cohort に追加すればよい